**01 - Data Profile**

*Purpose*: Inspect the Last.fm-1K raw files, understand schema, row counts, missing values, date ranges, and user/event volume.

This file will help us understand:

- number of raw files
- file sizes
- profile table shape
- listening sample shape
- columns
- missing values
- date range
- number of users
- number of artists
- number of tracks
- whether timestamps are usable
- whether user-level analytics is possible


# Importing Libraries

In [1]:
from pathlib import Path
import pandas as pd
from typing import List
import numpy as np

# Setup

In [2]:
import sys
sys.path.append("..")
from src.ingest import get_raw_files, load_listening_events_sample, load_user_profile, get_file_size_mb, profile_listening_events_in_chunks

# Raw File Inventory

In [3]:
files = get_raw_files()
print(files)


[WindowsPath('C:/Users/nairs/OneDrive/Documents/Surabhi/Projects/StreamPulse-Analytics/data/raw/lastfm_1k/userid-profile.tsv'), WindowsPath('C:/Users/nairs/OneDrive/Documents/Surabhi/Projects/StreamPulse-Analytics/data/raw/lastfm_1k/userid-timestamp-artid-artname-traid-traname.tsv')]


In [4]:
preview  = []
for f in files:
    path = Path(f)
    preview.append(
        {
            "file_name" : path.name,
            "file_path": path,
            "size_mb": get_file_size_mb(path)

        }
    )

In [5]:
preview_df = pd.DataFrame(preview)
preview_df

,file_name,file_path,size_mb
0,userid-profile.tsv,C:\Users\nairs\OneDrive\Documents\Surabhi\Proj...,0.04
1,userid-timestamp-artid-artname-traid-traname.tsv,C:\Users\nairs\OneDrive\Documents\Surabhi\Proj...,2412.03


# User Profile Data Preview



## Load the data

In [6]:
profile_data = load_user_profile()
profile_data.head()

,user_id,gender,age,country,signup_date
0,user_000001,m,NaN,Japan,"Aug 13, 2006"
1,user_000002,f,NaN,Peru,"Feb 24, 2006"
2,user_000003,m,22.0,United States,"Oct 30, 2005"
3,user_000004,f,NaN,NaN,"Apr 26, 2006"
4,user_000005,m,NaN,Bulgaria,"Jun 29, 2006"


## shape

In [7]:
profile_data.shape

(992, 5)

## column headers

In [8]:
profile_data.columns.to_list()

['user_id', 'gender', 'age', 'country', 'signup_date']

## df info

In [9]:
profile_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   user_id      992 non-null    str    
 1   gender       884 non-null    str    
 2   age          286 non-null    float64
 3   country      907 non-null    str    
 4   signup_date  984 non-null    str    
dtypes: float64(1), str(4)
memory usage: 70.8 KB


## missing values

In [10]:
profile_data.isna().sum()

user_id          0
gender         108
age            706
country         85
signup_date      8
dtype: int64

In [11]:
missing_profile_df = pd.DataFrame({
    "missing_count": profile_data.isna().sum(),
    "percentage" : (profile_data.isna().sum() / len(profile_data) ) * 100
}).sort_values(by='percentage', ascending=False)
missing_profile_df

,missing_count,percentage
age,706,71.169355
gender,108,10.887097
country,85,8.568548
signup_date,8,0.806452
user_id,0,0.000000


## basic age summary

In [12]:
profile_data["age"].describe()


count    286.000000
mean      25.367133
std        8.314952
min        3.000000
25%       21.000000
50%       23.000000
75%       28.000000
max      103.000000
Name: age, dtype: float64

## country value counts

In [13]:
profile_data.value_counts(profile_data["country"])

country
United States               228
United Kingdom              126
Poland                       50
Germany                      36
Norway                       35
                           ... 
India                         1
Tunisia                       1
Iceland                       1
Northern Mariana Islands      1
Bosnia and Herzegovina        1
Name: count, Length: 66, dtype: int64

# Listening Events Sample Preview

## load the data

In [14]:
listening_data = load_listening_events_sample()
listening_data.head()

,user_id,timestamp,artist_id,artist_name,track_id,track_name
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)
3,user_000001,2009-05-04T13:42:52Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15)
4,user_000001,2009-05-04T13:42:11Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15)


## shape

In [15]:
listening_data.shape

(100000, 6)

## columns

In [16]:
listening_data.columns.to_list()

['user_id', 'timestamp', 'artist_id', 'artist_name', 'track_id', 'track_name']

## info

In [17]:
listening_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   user_id      100000 non-null  str  
 1   timestamp    100000 non-null  str  
 2   artist_id    97229 non-null   str  
 3   artist_name  100000 non-null  str  
 4   track_id     88189 non-null   str  
 5   track_name   100000 non-null  str  
dtypes: str(6)
memory usage: 16.6 MB


## missing values

In [18]:
missing_listening_df = pd.DataFrame(

    {
        "listening_missing_count": listening_data.isna().sum(),
        "percentage" : (listening_data.isna().sum() / len(listening_data) ) * 100
    }
).sort_values(by = 'percentage', ascending=False)

missing_listening_df


,listening_missing_count,percentage
track_id,11811,11.811
artist_id,2771,2.771
timestamp,0,0.000
user_id,0,0.000
artist_name,0,0.000
track_name,0,0.000


## duplicate rows

In [19]:
listening_data[listening_data.duplicated() ==True]


,user_id,timestamp,artist_id,artist_name,track_id,track_name


# Basic Data Quality Checks

## missing user_id count

In [20]:
missing_listening_df.loc["user_id"]

listening_missing_count    0.0
percentage                 0.0
Name: user_id, dtype: float64

## missing timestamp count

In [21]:
missing_listening_df.loc["timestamp"]

listening_missing_count    0.0
percentage                 0.0
Name: timestamp, dtype: float64

## missing artist_name count

In [22]:
missing_listening_df.loc["artist_name"]

listening_missing_count    0.0
percentage                 0.0
Name: artist_name, dtype: float64


## missing track_name count

In [23]:
missing_listening_df.loc["track_name"]

listening_missing_count    0.0
percentage                 0.0
Name: track_name, dtype: float64

## duplicate row count


In [24]:
listening_data.duplicated().sum()

np.int64(0)

## unique users

In [25]:
listening_data["user_id"].unique()

<ArrowStringArray>
['user_000001', 'user_000002', 'user_000003', 'user_000004']
Length: 4, dtype: str

## unique artists

In [26]:
listening_data["artist_name"].unique()

<ArrowStringArray>
[          'Deep Dish',                '坂本龍一',          'Underworld',
     'Ennio Morricone',             'Minus 8',           'Beanfield',
            'Dj Linus',           'Alif Tree',             'Wei-Chi',
           'Marsmobil',
 ...
              'Blutch',      'Naoki Hasegawa',          'Ken Reaume',
       'Fortune Drive',       'Akira Yamaoka',       'Funeral Dress',
           'Telefilme',         'Dawn Landes',       'Nina Nastasia',
 'The Lightning Seeds']
Length: 3407, dtype: str

## unique tracks

In [27]:
listening_data["track_name"].unique()

<ArrowStringArray>
['Fuck Me Im Famous (Pacha Ibiza)-09-28-2007',
          'Composition 0919 (Live_2009_4_15)',
                       'Mc2 (Live_2009_4_15)',
                    'Hibari (Live_2009_4_15)',
                       'Mc1 (Live_2009_4_15)',
               'To Stanford (Live_2009_4_15)',
             'Improvisation (Live_2009_4_15)',
                   'Glacier (Live_2009_4_15)',
                 'Parolibre (Live_2009_4_15)',
            'Bibo No Aozora (Live_2009_4_15)',
 ...
                                  'The Devil',
                               'Rented Rooms',
                                  'Lucky You',
                             'Generation Sex',
                                  'Ice Cream',
                    'Hiding On The Staircase',
                                  'Get Lucky',
                                 'Get Dancey',
                                    'Jerk Me',
                                 'The Get Go']
Length: 16853, dtype: str

# Timestamp and Date Range Check

## Convert into datetime

In [28]:
listening_data["timestamp"] = pd.to_datetime(
    listening_data["timestamp"],
    errors="coerce"
)
listening_data

,user_id,timestamp,artist_id,artist_name,track_id,track_name
0,user_000001,2009-05-04 23:08:57+00:00,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04 13:54:10+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04 13:52:04+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)
3,user_000001,2009-05-04 13:42:52+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15)
4,user_000001,2009-05-04 13:42:11+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15)
...,...,...,...,...,...,...
99995,user_000004,2008-03-20 14:08:49+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7da1c0-108a-456a-baa2-e5d29e349324,Get Lucky
99996,user_000004,2008-03-20 14:04:57+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,27534cc1-6140-4c0a-82e2-dfc1b32bcdea,Tight Fit
99997,user_000004,2008-03-20 13:59:30+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,c271d600-46db-4f65-9231-dd5d334b12c8,Get Dancey
99998,user_000004,2008-03-20 13:55:22+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7732f4-40e2-40ab-b113-b3b73bab51b5,Jerk Me


## Store month, year, day and time

In [29]:
listening_data["month"] = listening_data['timestamp'].dt.month
listening_data["year"] = listening_data['timestamp'].dt.year
listening_data["time"] = listening_data["timestamp"].dt.time
listening_data["hour"] = listening_data["timestamp"].dt.hour
listening_data["month_name"] = listening_data['timestamp'].dt.month_name()
listening_data["day_of_week"] = listening_data["timestamp"].dt.day_of_week
listening_data["day_name"] = listening_data["timestamp"].dt.day_name()
listening_data["year_month"] = listening_data["timestamp"].dt.to_period("M")


C:\Users\nairs\AppData\Local\Temp\ipykernel_41736\2424703396.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  listening_data["year_month"] = listening_data["timestamp"].dt.to_period("M")


In [30]:
listening_data

,user_id,timestamp,artist_id,artist_name,track_id,track_name,month,year,time,hour,month_name,day_of_week,day_name,year_month
0,user_000001,2009-05-04 23:08:57+00:00,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,5,2009,23:08:57,23,May,0,Monday,2009-05
1,user_000001,2009-05-04 13:54:10+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),5,2009,13:54:10,13,May,0,Monday,2009-05
2,user_000001,2009-05-04 13:52:04+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),5,2009,13:52:04,13,May,0,Monday,2009-05
3,user_000001,2009-05-04 13:42:52+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15),5,2009,13:42:52,13,May,0,Monday,2009-05
4,user_000001,2009-05-04 13:42:11+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15),5,2009,13:42:11,13,May,0,Monday,2009-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,user_000004,2008-03-20 14:08:49+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7da1c0-108a-456a-baa2-e5d29e349324,Get Lucky,3,2008,14:08:49,14,March,3,Thursday,2008-03
99996,user_000004,2008-03-20 14:04:57+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,27534cc1-6140-4c0a-82e2-dfc1b32bcdea,Tight Fit,3,2008,14:04:57,14,March,3,Thursday,2008-03
99997,user_000004,2008-03-20 13:59:30+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,c271d600-46db-4f65-9231-dd5d334b12c8,Get Dancey,3,2008,13:59:30,13,March,3,Thursday,2008-03
99998,user_000004,2008-03-20 13:55:22+00:00,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7732f4-40e2-40ab-b113-b3b73bab51b5,Jerk Me,3,2008,13:55:22,13,March,3,Thursday,2008-03


## Min timestamp

In [31]:
min_timestamp = listening_data["timestamp"].min()
min_timestamp

Timestamp('2005-10-30 22:23:21+0000', tz='UTC')

## max timestamp

In [32]:
max_timestamp = listening_data["timestamp"].max()
max_timestamp

Timestamp('2009-05-04 23:08:57+0000', tz='UTC')

## number of years covered

In [33]:
year_min = listening_data["year"].min()
year_max = listening_data["year"].max()

print(f"Sample data ranges from the year {min_timestamp.date()} to {max_timestamp.date()}. \nTotal unique years in the sample: {year_max - year_min}")

Sample data ranges from the year 2005-10-30 to 2009-05-04. 
Total unique years in the sample: 4


## events by year

In [34]:
listening_data["year"].value_counts()

year
2006    35155
2008    32006
2007    23632
2009     8731
2005      476
Name: count, dtype: int64

## events by month


In [35]:
listening_data["month"].value_counts()

month
11    10289
10     9785
7      9635
3      9560
12     9067
8      8965
4      8648
9      8526
1      7823
2      7686
5      5548
6      4468
Name: count, dtype: int64

# Entity Counts


## events per user summary

In [36]:


listening_data["user_id"].value_counts()




user_id
user_000002    57438
user_000003    19494
user_000001    16685
user_000004     6383
Name: count, dtype: int64

## artists per user summary

In [37]:
unique_artists_per_user = listening_data.groupby("user_id").agg(
    unique_artists = ("artist_name", "nunique")
)
unique_artists_per_user

,unique_artists
user_id,
user_000001,657
user_000002,1283
user_000003,935
user_000004,995


## Top artists in sample

In [38]:
listening_data["artist_name"].value_counts()

artist_name
The Libertines         2411
Babyshambles           1724
Kettcar                1282
Death Cab For Cutie    1093
The Kooks               979
                       ... 
Funeral Dress             1
Telefilme                 1
Dawn Landes               1
Nina Nastasia             1
The Lightning Seeds       1
Name: count, Length: 3407, dtype: int64

## tracks per user summary

In [39]:
unique_tracks_per_user = listening_data.groupby("user_id").agg(
    unique_tracks = ("track_name", "nunique")
)
unique_tracks_per_user

,unique_tracks
user_id,
user_000001,3092
user_000002,8129
user_000003,4565
user_000004,2430


## top tracks in sample

In [40]:
listening_data["track_name"].value_counts()

track_name
Black Is The Colour                295
Wheelpusher                        222
Let Go                             154
Cannonball                         139
I Will Follow You Into The Dark    139
                                  ... 
Hiding On The Staircase              1
Get Lucky                            1
Get Dancey                           1
Jerk Me                              1
The Get Go                           1
Name: count, Length: 16853, dtype: int64

# Full Listening Dataset Profile

## Load the data summary

In [42]:
full_profile = profile_listening_events_in_chunks(chunksize=500_000)

## Total rows

In [43]:
full_profile["total_rows"]

19098853

## unique users

In [44]:
full_profile["unique_users"]

992

## unique artists

In [45]:
full_profile["unique_artists"]

173921

## unique tracks

In [46]:
full_profile["unique_tracks"]

1083470

## min timestamp

In [47]:
full_profile["min_timestamp"]

Timestamp('2005-02-14 00:00:07+0000', tz='UTC')

## max timestamp

In [48]:
full_profile["max_timestamp"]

Timestamp('2013-09-29 18:32:04+0000', tz='UTC')

## missing counts

In [49]:
full_profile["missing_counts"]

user_id              0
timestamp            0
artist_id       600848
artist_name          0
track_id       2162719
track_name         210
dtype: int64

## missing percentages

In [51]:
full_profile["missing_percentage"]

user_id         0.00
timestamp       0.00
artist_id       3.15
artist_name     0.00
track_id       11.32
track_name      0.00
dtype: float64

# Initial Observations

* The dataset is historical, with listening activity ranging from 2005 to 2013. Because of this, findings should be framed as a product analytics case study rather than a reflection of current music-streaming behavior. However, the dataset is still valuable because it provides enough user-level, timestamped listening activity to practice retention analysis, funnel analysis, segmentation, and predictive modeling.

* The full listening dataset contains timestamped events across multiple users, which makes user-level product analytics possible.

* Retention analysis appears feasible because the dataset contains listening timestamps over a long time period.

* The 100K-row sample is useful for initial schema inspection and quality checks, but it is not representative of the full dataset because the raw file appears to be ordered by `user_id`.

* The full dataset covers approximately 8.6 years of listening activity, from 2005-02-14 to 2013-09-29.

* Most core columns are highly complete. The main missingness appears in `artist_id` and `track_id`, while `track_name` has only 210 missing values, which is negligible compared to the full dataset size.

* Because `artist_id` and `track_id` have missing values, later analysis may need to use artist and track names as fallback identifiers. This decision should be documented carefully during the cleaning phase.

* Since the chunked reader uses `on_bad_lines="skip"`, malformed rows may be excluded from the parsed row count. This should be documented as a data quality limitation.


# Phase 1 Notes for Documentation

* Raw Last.fm-1K files were successfully detected in `data/raw/lastfm_1k/`.

* A 100K-row sample was used for quick schema inspection and initial quality checks.

* Full-dataset profiling was performed using chunked processing because the listening events file is large and should not be loaded into memory all at once.

* Full profiling confirmed that the dataset supports user-level analysis, retention analysis, replay modeling, segmentation, and funnel analysis.

* The dataset does not support saved-track analysis, playlist-add analysis, skip behavior analysis, device/platform analysis, premium conversion, revenue analysis, or causal A/B testing.

* Artist and track IDs have missing values, so future cleaning and modeling steps should decide whether to rely on IDs, names, or combined fallback keys.

* The dataset is historical, so final business recommendations should be presented as case-study recommendations rather than current market claims.
